# PostgreSQL Data Load — Verification Notebook
Connects to the PostgreSQL database and verifies all 5 tables loaded correctly after migration.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import warnings
warnings.filterwarnings('ignore')

DB_USER     = 'postgres'
DB_PASSWORD = 'CIpass2024@@'
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'carinsurance_db'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('Connected to PostgreSQL successfully!')

## 1. Table Row Counts

In [ ]:
tables = ['policyholders', 'vehicles', 'policies', 'claims', 'payments']
print(f'{"Table":20s} {"Row Count":>10s}')
print('-' * 32)
for t in tables:
    with engine.connect() as conn:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
    print(f'{t:20s} {count:>10,}')

## 2. Sample Data — Policyholders

In [ ]:
df = pd.read_sql('SELECT * FROM policyholders LIMIT 5', engine)
df

## 3. Sample Data — Claims

In [ ]:
df = pd.read_sql('SELECT * FROM claims LIMIT 5', engine)
df

## 4. Migration Health Check

In [ ]:
query = '''
    SELECT
        (SELECT COUNT(*) FROM policyholders)                             AS total_policyholders,
        (SELECT COUNT(*) FROM policies)                                  AS total_policies,
        (SELECT COUNT(*) FROM claims)                                    AS total_claims,
        (SELECT COUNT(*) FROM claims WHERE is_fraud_flag = TRUE)         AS fraud_claims,
        (SELECT COUNT(*) FROM payments WHERE is_late = TRUE)             AS late_payments,
        (SELECT ROUND(AVG(premium_usd)::numeric,2) FROM policies)        AS avg_premium,
        (SELECT ROUND(AVG(credit_score)::numeric,1) FROM policyholders)  AS avg_credit_score
'''
result = pd.read_sql(query, engine)
result.T.rename(columns={0: 'Value'})

## 5. High Risk Policyholders by State

In [ ]:
query = '''
    SELECT state,
           COUNT(*) AS high_risk_count,
           ROUND(AVG(credit_score)::numeric, 1) AS avg_credit_score
    FROM policyholders
    WHERE credit_score < 550 OR prior_claims_count >= 5
    GROUP BY state
    ORDER BY high_risk_count DESC
    LIMIT 10
'''
df = pd.read_sql(query, engine)
print('Top 10 States by High-Risk Policyholders:')
df

## 6. Fraud Analysis by Claim Type

In [ ]:
query = '''
    SELECT claim_type,
           COUNT(*) AS total,
           SUM(CASE WHEN is_fraud_flag THEN 1 ELSE 0 END) AS fraud_count,
           ROUND(AVG(claim_amount_usd)::numeric, 2) AS avg_amount
    FROM claims
    GROUP BY claim_type
    ORDER BY fraud_count DESC
'''
df = pd.read_sql(query, engine)
df

## 7. Payment Method Breakdown

In [ ]:
query = '''
    SELECT payment_method,
           COUNT(*) AS total_payments,
           SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS late_count,
           ROUND(AVG(amount_usd)::numeric, 2) AS avg_amount
    FROM payments
    GROUP BY payment_method
    ORDER BY total_payments DESC
'''
df = pd.read_sql(query, engine)
df